# Model Quantization with bitsandbytes

**Quantization** reduces the numerical precision used to store a model's weights. This shrinks the memory footprint and can speed up inference, usually with only a small loss in quality — which is what makes it possible to run large language models on modest GPUs.

In this notebook we load the same model, **`meta-llama/Llama-3.2-1B-Instruct`**, at four different precisions and compare them:

| Precision | dtype | Bytes / weight | Typical use |
|-----------|----------|----------------|-------------|
| 32-bit | `float32` | 4 | Full precision (baseline) |
| 16-bit | `bfloat16` | 2 | Standard for training/inference on modern GPUs |
| 8-bit | `int8` | ~1 | Large memory savings, minimal quality loss |
| 4-bit | `NF4` | ~0.5 | Maximum savings, ideal for big models on small GPUs |

**What this notebook walks through:**

1. **Setup** — install the required libraries.
2. **Authentication** — log in to the Hugging Face Hub using a token stored in a `.env` file.
3. **Loading** — load the model at all four precisions.
4. **Size comparison** — measure each variant's in-memory footprint.
5. **Weight inspection** — see how the raw weight values change (floats vs. integers).
6. **Value ranges** — compare the min/max of the weights.
7. **Architecture** — see how the layer types differ per precision.
8. **GPU memory** — measure the real GPU memory used.

> **Requirements:** a GPU runtime (e.g. Google Colab with a GPU) and a Hugging Face account that has accepted the [Llama 3.2 license](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct).

## 1. Setup & installation

Install the required Python packages. In Google Colab this only needs to be run once per session.

In [1]:
# Install the core libraries used throughout this notebook.
#   transformers    -> loading models and running inference
#   bitsandbytes    -> the 8-bit and 4-bit quantization backends
#   accelerate      -> automatic device placement (device_map="auto")
#   trl / peft      -> fine-tuning helpers (used in the later notebooks)
#   datasets        -> dataset loading
#   huggingface_hub -> authentication and model downloads
#   python-dotenv   -> reads the Hugging Face token from a local .env file
!pip install -q -U transformers bitsandbytes accelerate trl peft datasets huggingface_hub python-dotenv

## 2. Authenticate with Hugging Face

Instead of the interactive `login()` prompt, we read the token from a local **`.env`** file. Create a file named `.env` next to this notebook and add your token:

```text
HF_TOKEN=hf_your_token_here
```

Generate a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). Keeping it in `.env` (which is git-ignored) avoids hard-coding secrets in the notebook. In Google Colab, upload the `.env` file to the session's working directory.

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load environment variables from the local .env file.
# Create a .env file next to this notebook containing a single line:
#   HF_TOKEN=hf_your_token_here
load_dotenv()

hf_token = os.getenv("HF_TOKEN")
if not hf_token or hf_token == "your_hugging_face_token_here":
    raise ValueError(
        "HF_TOKEN not found. Create a .env file next to this notebook with a "
        "line like: HF_TOKEN=hf_your_token_here (get a token at "
        "https://huggingface.co/settings/tokens)."
    )

# Authenticate with the Hugging Face Hub so we can download gated models such as
# Llama 3.2 (you must first accept its license on the model's Hub page).
login(token=hf_token)
print("Successfully authenticated with the Hugging Face Hub.")

Successfully authenticated with the Hugging Face Hub.


## 3. Load the model at four precisions

We download `Llama-3.2-1B-Instruct` and instantiate it four times at different precisions. `device_map="auto"` lets `accelerate` place the weights on the available GPU automatically.

> The first run downloads the model weights (~2.5 GB) and may take a couple of minutes.

In [3]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# We load the SAME model at four different numerical precisions so we can
# compare their memory footprint and see how quantization changes the weights.
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# Quantization configs provided by bitsandbytes:
#   8-bit -> each weight stored as int8  (~1 byte per weight)
#   4-bit -> each weight stored in 4 bits (~0.5 byte per weight, NF4 by default)
quantization_config_8bit = BitsAndBytesConfig(load_in_8bit=True)
quantization_config_4bit = BitsAndBytesConfig(load_in_4bit=True)

# 4-bit quantized model (smallest memory footprint).
print("Loading 4-bit model...")
model_4bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config_4bit,
    device_map="auto",
)

# 8-bit quantized model.
print("Loading 8-bit model...")
model_8bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config_8bit,
    device_map="auto",
)

# 16-bit model (bfloat16): half precision, no quantization.
print("Loading 16-bit (bfloat16) model...")
model_16bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# 32-bit model (float32): full precision, the default, largest footprint.
print("Loading 32-bit (float32) model...")
model_32bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)

print("All four model variants loaded successfully.")


Loading 4-bit model...
Loading 8-bit model...
Loading 16-bit (bfloat16) model...
Loading 32-bit (float32) model...
All four model variants loaded successfully.


## 4. Compare model sizes

The helper below sums the bytes used by every parameter and buffer. Notice how the footprint roughly halves as we move from 32 → 16 → 8 → 4 bits.

In [4]:
import torch


def print_model_size_in_mb_gb(model, label=""):
    """Print the total size of a PyTorch model (parameters + buffers) in MB and GB."""
    # Sum the bytes used by every parameter tensor.
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    # Buffers hold non-trainable tensors (e.g. running statistics); include them too.
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())

    size_bytes = param_size + buffer_size
    size_mb = size_bytes / (1024 ** 2)
    size_gb = size_mb / 1024

    prefix = f"{label}: " if label else ""
    print(f"{prefix}Total model size: {size_mb:.2f} MB ({size_gb:.2f} GB)")


# Compare the in-memory size of each precision.
# Lower precision => fewer bytes per weight => smaller model.
print_model_size_in_mb_gb(model_4bit, "4-bit ")
print_model_size_in_mb_gb(model_8bit, "8-bit ")
print_model_size_in_mb_gb(model_16bit, "16-bit")
print_model_size_in_mb_gb(model_32bit, "32-bit")

4-bit : Total model size: 965.13 MB (0.94 GB)
8-bit : Total model size: 1429.13 MB (1.40 GB)
16-bit: Total model size: 2357.13 MB (2.30 GB)
32-bit: Total model size: 4714.26 MB (4.60 GB)


## 5. Inspect the raw weight values

Quantization changes *how* the weights are stored. Below we print the same weight matrix (`q_proj` in the first decoder layer) from the 32-bit and 8-bit models. The 32-bit version holds floating-point numbers, while the 8-bit version holds small integers.

In [5]:
import torch

# Print full decimal numbers (not scientific notation) so the raw values are readable.
torch.set_printoptions(precision=8, sci_mode=False)

# Inspect the query-projection (q_proj) weight matrix of the first decoder layer.
layer_32bit = model_32bit.model.layers[0]
weight_32bit = layer_32bit.self_attn.q_proj.weight

# In 32-bit, the weights are stored as floating-point numbers.
print("32-bit weight values (float32):")
print(weight_32bit)

32-bit weight values (float32):
Parameter containing:
tensor([[-0.01794434,  0.00662231,  0.02465820,  ..., -0.00872803,
         -0.01171875,  0.02014160],
        [ 0.01220703,  0.05932617,  0.05517578,  ..., -0.03320312,
         -0.01538086,  0.01080322],
        [ 0.01782227,  0.01550293,  0.03442383,  ..., -0.03857422,
         -0.03857422, -0.02758789],
        ...,
        [ 0.02978516,  0.03515625,  0.07128906,  ..., -0.07177734,
         -0.02648926, -0.02868652],
        [ 0.02258301, -0.02478027,  0.03515625,  ..., -0.01196289,
         -0.02868652, -0.01477051],
        [-0.02575684, -0.05371094, -0.01306152,  ...,  0.05419922,
          0.00958252, -0.00277710]], device='cuda:0', requires_grad=True)


In [6]:
import torch

torch.set_printoptions(precision=8, sci_mode=False)

# The same weight matrix as the previous cell, but from the 8-bit quantized model.
layer_8bit = model_8bit.model.layers[0]
weight_8bit = layer_8bit.self_attn.q_proj.weight

# In 8-bit, the same weights are stored as integers (int8) rather than floats.
print("8-bit weight values (int8):")
print(weight_8bit)

8-bit weight values (int8):
Parameter containing:
Parameter(Int8Params([[-44,  16,  61,  ..., -22, -29,  50],
            [ 14,  70,  65,  ..., -39, -18,  13],
            [ 16,  14,  30,  ..., -34, -34, -24],
            ...,
            [ 17,  20,  40,  ..., -40, -15, -16],
            [ 32, -35,  50,  ..., -17, -41, -21],
            [-14, -30,  -7,  ...,  30,   5,  -2]], device='cuda:0',
           dtype=torch.int8))


## 6. Weight value ranges

Comparing the min/max of the weights makes the storage difference concrete: the 32-bit floats span a small decimal range (roughly -0.7 to 0.6), while the int8 values span the full integer range (roughly -127 to +127).

In [7]:
# Value range of the raw 32-bit floating-point weights.
layer_32bit = model_32bit.model.layers[0]
weight_max = layer_32bit.self_attn.q_proj.weight.max()
weight_min = layer_32bit.self_attn.q_proj.weight.min()

print("32-bit weight range (float32):")
print(f"  max = {weight_max}")
print(f"  min = {weight_min}")

32-bit weight range (float32):
  max = 0.58203125
  min = -0.67578125


In [8]:
# Value range of the same weights after 8-bit quantization.
# The float values are mapped onto the int8 range (roughly -127 .. +127).
layer_8bit = model_8bit.model.layers[0]
weight_max = layer_8bit.self_attn.q_proj.weight.max()
weight_min = layer_8bit.self_attn.q_proj.weight.min()

print("8-bit weight range (int8):")
print(f"  max = {weight_max}")
print(f"  min = {weight_min}")

8-bit weight range (int8):
  max = 127
  min = -127


## 7. Inspect the model architecture

Printing each model reveals which layer types back the projections at every precision:

- **32-bit / 16-bit** → standard `Linear`
- **8-bit** → `Linear8bitLt`
- **4-bit** → `Linear4bit`

The overall structure (16 decoder layers, attention + MLP blocks) is identical; only the linear layers change.

In [9]:
# Architecture of the 32-bit model. Note the standard `Linear` projection layers.
model_32bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb):

In [10]:
# 8-bit model: the attention/MLP projections are now `Linear8bitLt` layers.
model_8bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMS

In [11]:
# 4-bit model: the projections become `Linear4bit` layers.
model_4bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), 

In [12]:
# 16-bit model: layers stay standard `Linear`, but the weights are stored as bfloat16.
model_16bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb):

## 8. Measure real GPU memory usage

The sizes computed earlier are the *theoretical* weight sizes. Here we measure the **actual** GPU memory consumed when loading the 8-bit model, which also accounts for allocation overhead.

In [13]:
import torch

# Measure how much GPU memory the 8-bit model actually occupies once loaded.
# We clear the cache first, record the baseline, load the model, then compare.
torch.cuda.empty_cache()
start_mem = torch.cuda.memory_allocated(device=0)

model_8bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config_8bit,
    device_map="auto",
)

end_mem = torch.cuda.memory_allocated(device=0)
used_mem = end_mem - start_mem
print(f"8-bit model GPU memory usage: {used_mem / 1024**2:.2f} MB")

8-bit model GPU memory usage: 1431.57 MB
